In [2]:
import tifffile as tiff
import numpy as np
from spatialdata import SpatialData
from spatialdata.models import Image2DModel, PointsModel
import harpy as hp
import pandas as pd
from pathlib import Path
import spatialdata_plot
from napari_spatialdata import Interactive

import torch
from cellpose import models
from harpy.image import cellpose_callable

# Different sections:
### 1. Reading in the data
### 2. Image processing
### 3. Cell segmentation
### 4. Transcript allocation
### 5. Single-cell analysis: 
##### 5.1 filtering, normalization and qc
##### 5.2 clustering
##### 5.3 cell type annotation
##### 5.4 

## Reading in the data

In [ ]:
file_path = "/Volumes/Intenso/spatial-transcriptomics/"

In [ ]:
# loading the image and parsing it 
img = tiff.imread(f"{file_path}A1_DAPI.tiff")  # shape likely (y, x) or (c, y, x)
if img.ndim == 2:
    img = img[None, ...]

image = Image2DModel.parse(img, dims=("c", "y", "x"))

In [ ]:
# loading the transcripts data
df = pd.read_csv(
    f"{file_path}A1_results.txt",
    sep=r"\s+",
    header=None,
    names=["x", "y", "z", "gene"],
    engine="python",
)
df.head(10)

In [ ]:
points = PointsModel.parse(df, coordinates={"x": "x", "y": "y"})

In [ ]:
# creating the sdata object
sdata = SpatialData(
    images={"a1_dapi": image},
    points={"a1_genes": points},
)
sdata

In [ ]:
sdata.pl.render_images("a1_dapi", cmap = "gray").pl.show()

In [ ]:
hp.pl.plot_image(
    sdata, 
    img_layer = "a1_dapi" , 
    crd = [6000, 12000, 6000, 9000], # (xmin, xmax, ymin, ymax). 
    figsize = (10,10),
)

In [ ]:
sdata["a1_genes"].compute()

In [ ]:
# write the object to zarr
# #sdata.write("/Volumes/Intenso/spatial-transcriptomics/intermediate_results/20251223.zarr", overwrite=True)

## Image processing

In [ ]:
sdata = hp.im.min_max_filtering(
    sdata,
    img_layer = "a1_dapi",
    output_layer = "a1_min_max_filtered",
    size_min_max_filter = 51,
    overwrite = True,
)

hp.pl.plot_image(sdata, img_layer=[ "a1_dapi", "a1_min_max_filtered" ], crd = [6000, 8000, 6000, 8000], figsize=(20,20))

In [ ]:
sdata = hp.im.enhance_contrast(
    sdata,
    img_layer = "a1_min_max_filtered",
    output_layer = "a1_clahe",
    contrast_clip = 20,
    chunks = 20000,
    overwrite = True
)

# Plot the contrast enhanced image
hp.pl.plot_image(sdata, img_layer=[ "a1_min_max_filtered", "a1_clahe" ], crd = [6000, 8000, 6000, 8000], figsize=(20,20))

In [ ]:
sdata

## Segmentation

In [ ]:
# from dask.distributed import Client, LocalCluster

# # # Create a local Dask cluster
# cluster = LocalCluster(
#      n_workers=1,              # Number of worker processes
#      threads_per_worker=4,    # Number of threads per worker
#      processes=False,
#      memory_limit="12GB",      # Memory limit per worker
#  )

# # # Connect a Client to the cluster
# client = Client(cluster)

# # # Print the Dask dashboard link
# print(client.dashboard_link)

In [ ]:
unit_testing = True 
gpu = False
device = "cpu"  # mps broken in cellpose (macOS), see https://github.com/MouseLand/cellpose/issues/1063

# Perform nucleus segmentation
sdata = hp.im.segment(
    sdata,
    img_layer="a1_clahe", # The image layer in sdata to be segmented.
    chunks=4096, #settings chunks=None would be equivalent to settings chunks=2048, as chunks on disk are 2048
    depth=32,
    model=cellpose_callable,
    # parameters that will be passed to the callable _cellpose:
    pretrained_model="nuclei", # can also be "cyto", "cyto3", or a path to a fine-tuned cellpose model.
    device=device,
    diameter=80,
    flow_threshold=0.6,
    cellprob_threshold=-6,
    min_size=40,
    output_labels_layer="a1_segmentation_mask",
    output_shapes_layer="a1_segmentation_mask_boundaries",
    crd=[6000, 8000, 6000, 8000] if unit_testing else None,  # region to segment [x_min, xmax, y_min, y_max],
    overwrite=True,
)


In [ ]:
# client.close()
# cluster.close()


In [ ]:
sdata

In [ ]:
hp.pl.plot_image(
    sdata, 
    img_layer="a1_clahe", 
    crd = [6000, 8000, 6000, 8000], 
    figsize=(10,10)
)

In [ ]:
import spatialdata_plot
hp.pl.plot_shapes(
    sdata, 
    img_layer="a1_clahe", 
    shapes_layer="a1_segmentation_mask_boundaries", 
    figsize=(10,10), 
    crd = [6000, 8000, 6000, 8000]
)

In [ ]:
hp.pl.plot_shapes(
    sdata, 
    img_layer="a1_clahe", 
    shapes_layer="a1_segmentation_mask_boundaries", 
    figsize=(10,10), 
    crd = [6000, 8000, 6000, 8000]
)